In [1]:
import pandas as pd
import optuna
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_val_score

In [3]:
# Load the dataset
df = pd.read_csv(r'C:\Users\khiew\Downloads\FYP Reduced Dataset.csv')

In [4]:
# Group diseases with <100 samples into "Other"
disease_counts = df['diseases'].value_counts()
df['diseases'] = df['diseases'].apply(lambda x: x if disease_counts[x] >= 100 else 'Other')

In [5]:
# Assuming that the target variable is 'diseases' and all other variables are input features
X = df.drop('diseases', axis=1)
y = df['diseases']

# Encode the target variable (diseases) if it's a categorical variable
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

In [7]:
# Optuna optimization function
def objective(trial):
    # Define the hyperparameters to tune
    n_estimators = trial.suggest_int('n_estimators', 50, 150)
    max_depth = trial.suggest_int('max_depth', 10, 50)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 20)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 20)
    max_features = trial.suggest_categorical('max_features', ['sqrt', 'log2', None])

    # Create RandomForestClassifier with hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        random_state=42
    )

    # Perform cross-validation (using 5-fold by default)
    score = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
    accuracy = score.mean()

    # Log the hyperparameters and accuracy for this trial
    print(f"Trial {trial.number}: n_estimators={n_estimators}, max_depth={max_depth}, "
          f"min_samples_split={min_samples_split}, min_samples_leaf={min_samples_leaf}, "
          f"max_features={max_features}, Accuracy={accuracy:.4f}")

    return accuracy

In [9]:
# Create Optuna study for optimization with persistent storage
study = optuna.create_study(
    direction='maximize',  # Assuming you are maximizing accuracy
    study_name="randomforest_diseases_symptoms_other_study", 
    storage=r"sqlite:///C:/Users/khiew/Downloads/randomforest.db", 
    load_if_exists=True  # Load the study if it already exists, to resume from the last trial
)

# Optimize the study with your objective function, you can adjust the n_trials as needed
study.optimize(objective, n_trials=20)
# Print the best trial and hyperparameters
print("\nBest Trial:")
print(study.best_trial)
print("Best Hyperparameters:")
print(study.best_trial.params)

[I 2025-04-20 22:42:44,306] A new study created in RDB with name: randomforest_diseases_symptoms_other_study
[I 2025-04-20 22:44:44,528] Trial 0 finished with value: 0.339913731585478 and parameters: {'n_estimators': 143, 'max_depth': 45, 'min_samples_split': 19, 'min_samples_leaf': 3, 'max_features': 'log2'}. Best is trial 0 with value: 0.339913731585478.


Trial 0: n_estimators=143, max_depth=45, min_samples_split=19, min_samples_leaf=3, max_features=log2, Accuracy=0.3399


[I 2025-04-20 22:46:02,152] Trial 1 finished with value: 0.33923544356381585 and parameters: {'n_estimators': 91, 'max_depth': 33, 'min_samples_split': 15, 'min_samples_leaf': 1, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.339913731585478.


Trial 1: n_estimators=91, max_depth=33, min_samples_split=15, min_samples_leaf=1, max_features=sqrt, Accuracy=0.3392


[I 2025-04-20 22:49:10,442] Trial 2 finished with value: 0.270191741924148 and parameters: {'n_estimators': 129, 'max_depth': 19, 'min_samples_split': 17, 'min_samples_leaf': 3, 'max_features': None}. Best is trial 0 with value: 0.339913731585478.


...

Trial 17: n_estimators=50, max_depth=25, min_samples_split=18, min_samples_leaf=1, max_features=log2, Accuracy=0.3352


[I 2025-04-20 23:13:46,847] Trial 18 finished with value: 0.33112634193858137 and parameters: {'n_estimators': 87, 'max_depth': 38, 'min_samples_split': 10, 'min_samples_leaf': 20, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.339913731585478.


Trial 18: n_estimators=87, max_depth=38, min_samples_split=10, min_samples_leaf=20, max_features=sqrt, Accuracy=0.3311


[I 2025-04-20 23:16:31,472] Trial 19 finished with value: 0.31253921119785344 and parameters: {'n_estimators': 100, 'max_depth': 26, 'min_samples_split': 4, 'min_samples_leaf': 6, 'max_features': None}. Best is trial 0 with value: 0.339913731585478.


Trial 19: n_estimators=100, max_depth=26, min_samples_split=4, min_samples_leaf=6, max_features=None, Accuracy=0.3125

Best Trial:
FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.339913731585478], datetime_start=datetime.datetime(2025, 4, 20, 22, 42, 44, 312967), datetime_complete=datetime.datetime(2025, 4, 20, 22, 44, 44, 502710), params={'n_estimators': 143, 'max_depth': 45, 'min_samples_split': 19, 'min_samples_leaf': 3, 'max_features': 'log2'}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=150, log=False, low=50, step=1), 'max_depth': IntDistribution(high=50, log=False, low=10, step=1), 'min_samples_split': IntDistribution(high=20, log=False, low=2, step=1), 'min_samples_leaf': IntDistribution(high=20, log=False, low=1, step=1), 'max_features': CategoricalDistribution(choices=('sqrt', 'log2', None))}, trial_id=23, value=None)
Best Hyperparameters:
{'n_estimators': 143, 'max_depth': 45, 'min_samples_split': 19,